# nudge-engine: set up, call, test, measure

Run the cells top to bottom. Each part is one cell. If a cell goes red, stop and paste the output to Claude.
Everything heavy runs on Modal, so you do not need a Colab GPU. **The models bill by the hour while they are awake. Run the last cell (Stop the billing) when you finish.**

| Part | What it does | Rough cost |
|---|---|---|
| 1 to 3 | get the code, load your keys, log in to Modal | free |
| **SETUP** | deploy the voice, brain and API, wake them, check them | the first wake-up is about 10 minutes of GPU |
| **CALL** | a pretend borrower speaks and the engine answers out loud | seconds |
| **EVALS** | 8 scripted borrowers, 5 runs each: do the rules hold? | about 10 minutes of GPU |
| **BENCHMARK** | how fast and how expensive: brain and voice | about 15 minutes of GPU |
| **COLD START** | puts the models to sleep and times 5 wake-ups | about an hour of GPU |
| **REPORT** | turns all the saved results into tables | free |
| **STOP** | lets the models sleep so billing stops | free |

**Before you start**
1. Push `nudge-engine` to your GitHub (a private repo is fine).
2. On Modal: add a payment method (GPUs need one) and set a monthly budget under Usage and Billing.
3. Open the key icon in Colab's left bar (Secrets), switch on *Notebook access*, and add these:

| Secret | Where it comes from |
|---|---|
| `DATABASE_URL` | Neon: connection string, edited to start with `postgresql+psycopg://` |
| `API_KEY` | make a long random string (step 2 prints how) |
| `MODELS_API_KEY` | make another long random string |
| `DEEPGRAM_API_KEY` | Deepgram console, role *Member* |
| `HF_TOKEN` | Hugging Face, a *Read* token |
| `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET` | Modal, Settings, API Tokens |
| `GITHUB_TOKEN` | only if the repo is private |
| `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, `TWILIO_FROM_NUMBER`, `ALLOWED_NUMBERS` | only for phone calls, at the very end |

In [ ]:
#@title 1. Get the code and install it
REPO_URL = "https://github.com/aidyai/nudge-engine.git"  #@param {type:"string"}

import os
from google.colab import userdata

def secret(name, default=""):
    try:
        return userdata.get(name) or default
    except Exception:  # not set, or this notebook was not given access
        return default

token = secret("GITHUB_TOKEN")  # only needed for a private repo
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

!pip install -q uv
if not os.path.exists("/content/nudge-engine"):
    !git clone -q {clone_url} /content/nudge-engine
%cd /content/nudge-engine
os.environ["PYTHONPATH"] = "/content/nudge-engine"  # every command below can import engine, evals and scripts
!uv sync -q
print("ready")

In [ ]:
#@title 2. Put your keys into a .env file (read from Colab Secrets)
import pathlib

NEEDED = ["DATABASE_URL", "API_KEY", "MODELS_API_KEY", "DEEPGRAM_API_KEY", "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
OPTIONAL = ["LLM_BASE_URL", "TTS_URL", "LLM_PROXY_KEY", "LLM_PROXY_SECRET", "TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN", "TWILIO_FROM_NUMBER", "ALLOWED_NUMBERS", "LANGFUSE_BASE_URL"]
KEEP_OUT_OF_ENV = {"MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"}  # used only to log in, never passed on to the app

missing = [n for n in NEEDED if not secret(n)]
if missing:
    print("To make a random string:  python -c \"import secrets; print(secrets.token_urlsafe(32))\"")
    raise SystemExit("Add these in Colab Secrets (key icon), switch on Notebook access, then run this cell again: " + ", ".join(missing))

values = {n: secret(n) for n in NEEDED + OPTIONAL if n not in KEEP_OUT_OF_ENV}
values["LLM_MODEL"] = "phonellm-mini-alpha-2"
pathlib.Path(".env").write_text("".join(f"{k}={v}\n" for k, v in values.items() if v))

def read_env():
    return dict(line.split("=", 1) for line in pathlib.Path(".env").read_text().splitlines() if "=" in line)

def set_env(key, value):
    env = read_env()
    env[key] = value
    pathlib.Path(".env").write_text("".join(f"{k}={v}\n" for k, v in env.items()))
    print(f"saved {key} to .env. Add it to Colab Secrets too, so you do not have to redeploy after a restart.")

print("keys found:", ", ".join(k for k, v in values.items() if v))

In [ ]:
#@title 3. Log in to Modal
modal_id, modal_secret = secret("MODAL_TOKEN_ID"), secret("MODAL_TOKEN_SECRET")
!uv run modal token set --token-id {modal_id} --token-secret {modal_secret}
!uv run modal profile current

## VOICE (optional): use a cloned voice

The voice speaks in the model's default voice unless a clip is saved in the Modal volume `nudge-voices` as `voice.wav`. Use about 10 seconds of clear speech from someone who agreed to be cloned. Run this once, before SETUP. It stays in the volume, so every start uses it.


In [ ]:
#@title VOICE: upload a clip (skip this cell to keep the default voice)
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
!uv run modal volume create nudge-voices || true
!uv run modal volume put --force nudge-voices "{name}" /voice.wav


## SETUP

One press does it all: the database, the secrets, the voice, the brain, the API, wakes the models, and checks them. The wake-up is the only long wait (about 10 minutes the first time).

Tick `SNAPSHOT` for the fast-wake brain (Modal GPU memory snapshot, still an early feature). It asks you once for a Modal proxy token, made at https://modal.com/settings/proxy-auth-tokens. Untick `SEED_DATABASE` on later runs.

In [ ]:
#@title SETUP: deploy everything, wake it, check it
SNAPSHOT = False  #@param {type:"boolean"}
SEED_DATABASE = True  #@param {type:"boolean"}
import os, subprocess

def stream(cmd):
    """Run a command and show its output live in the cell (Colab hides output of plain subprocess.run)."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

if SNAPSHOT and not (read_env().get("LLM_PROXY_KEY") and read_env().get("LLM_PROXY_SECRET")):
    from getpass import getpass
    print("Snapshot mode needs a Modal proxy token: https://modal.com/settings/proxy-auth-tokens")
    set_env("LLM_PROXY_KEY", getpass("Token ID (wk-...): ").strip())
    set_env("LLM_PROXY_SECRET", getpass("Token Secret (ws-...): ").strip())

cmd = ["uv", "run", "python", "-m", "scripts.launch"] + (["--snapshot"] if SNAPSHOT else []) + (["--seed"] if SEED_DATABASE else [])
if stream(cmd):
    raise SystemExit("Stopped. Paste the lines above to Claude.")

**You should see:** `All good.` then `Done. The models are awake and billing.`

## CALL: a phone call without the phone

A pretend borrower speaks (the voice model), Deepgram hears it, the agent and brain answer, and the voice speaks the answer. This goes through the same services as a real call. It prints the conversation, how long the borrower waits after each turn (ears + brain + voice), and saves `simulated_call.wav` with the agent's side.

In [ ]:
#@title CALL: simulate one full call, voice in and voice out
!uv run --env-file .env python -m scripts.simulate_call --label call

import os
from IPython.display import Audio
if os.path.exists("simulated_call.wav"):
    display(Audio("simulated_call.wav"))

**You should see:** the agent greeting, four borrower turns with what Deepgram heard, `outcome: promise_captured`, the silence times, and an audio player.

## LIVE CALL: the streaming call, like the browser

A pretend borrower streams its voice over the websocket in real time. The agent answers with its voice. This is what the website's demo uses, and it measures the silence after the borrower stops (the voice detector, smart-turn, ears, brain and voice together). It needs the models awake (SETUP does that).


In [ ]:
#@title LIVE CALL: simulate a streaming call (voice in, voice out)
!uv run --env-file .env python -m scripts.simulate_live --label live


## PROBE: the backchannel detector (once)

Prints what Pipecat's backchannel model takes in and gives out. Send me the output and I will plug it into the live call.


In [ ]:
#@title PROBE: print the backchannel model's inputs and outputs
!uv run python -m scripts.probe_backchannel


## EVALS: do the rules hold?

Eight scripted borrowers (a happy one, a wrong birthday, someone who says stop, someone who lost their job, someone who tries to trick the agent, and more), 5 runs each. Rules are things that must never break. Task is whether the call reached the right end. The cell stops with an error if any rule breaks.

In [ ]:
#@title EVALS: 8 scripted borrowers, 5 runs each
EVAL_LABEL = "baseline"  #@param {type:"string"}
RUNS = 5  #@param {type:"integer"}
!uv run modal secret create nudge-env --from-dotenv .env --force
!uv run --env-file .env modal run evals/run.py --runs {RUNS} --label {EVAL_LABEL}

## BENCHMARK: how fast, and what it costs

The models must be awake (SETUP does that). Prices are from modal.com/pricing on 2026-11-17: RTX PRO 6000 $0.000842 per second ($3.03 per hour), A10 $0.000306 per second ($1.10 per hour). Check them if you run this later. The cell refuses a price of 0, so the cost columns can never be fake.

In [ ]:
#@title BENCHMARK: the brain (1 to 16 calls at once) and the voice (1 to 4 at once)
BRAIN_PRICE_PER_HOUR = 3.03  #@param {type:"number"}
VOICE_PRICE_PER_HOUR = 1.10  #@param {type:"number"}
assert BRAIN_PRICE_PER_HOUR > 0 and VOICE_PRICE_PER_HOUR > 0, "Prices must be above 0: see modal.com/pricing"
!uv run --env-file .env modal run evals/bench.py::llm --label rtx6000-bf16 --gpu RTX-PRO-6000 --price {BRAIN_PRICE_PER_HOUR}
!uv run --env-file .env modal run evals/bench.py::tts --label a10 --gpu A10 --price {VOICE_PRICE_PER_HOUR}

## COLD START: how long a sleeping brain takes to wake

Press play. It lets the models sleep, then wakes the brain `RUNS` times, waiting 200 seconds before each so it is really asleep, and times each wake-up. Run it once per kind of deploy: with `SNAPSHOT` off (`LABEL = cold-plain`), then set up again with it on and run it with `LABEL = cold-snapshot`. The first snapshot wake-up makes the snapshot, so it is slow. The rest are the real number. About an hour of GPU if the wake-ups stay slow.

In [ ]:
#@title COLD START: wake the brain 5 times and time each one
LABEL = "cold-plain"  #@param {type:"string"}
NOTE = ""  #@param {type:"string"}
RUNS = 5  #@param {type:"integer"}
import os, subprocess

def stream(cmd):
    """Run a command and show its output live in the cell (Colab hides output of plain subprocess.run)."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

# 1) let everything sleep, 2) refresh the secret the timer reads, 3) wake the brain RUNS times and time each one
for c in (["uv", "run", "modal", "run", "deploy/warm.py::off"],
          ["uv", "run", "modal", "secret", "create", "nudge-env", "--from-dotenv", ".env", "--force"],
          ["uv", "run", "--env-file", ".env", "modal", "run", "evals/bench.py::cold", "--label", LABEL, "--runs", str(RUNS), "--note", NOTE]):
    if stream(c):
        raise SystemExit("Stopped. Paste the lines above to Claude.")
print("Done. Run the tables cell (step 15) to see the cold starts.")

## REPORT

Turns every saved result into tables (`docs/performance.md`): the simulated call, the evals, the brain and voice speeds, and the cold starts.

In [ ]:
#@title REPORT: make the tables
!uv run python -m evals.report --write
from IPython.display import Markdown
Markdown(open("docs/performance.md").read())

## STOP: stop the billing

Run this when you finish. It lets the brain and the voice sleep (billing stops about 2 minutes later). Nothing is deleted: the apps stay deployed and the next request wakes them.

In [ ]:
#@title STOP: let the models sleep
!uv run modal run deploy/warm.py::off

## Later: phone calls (only after Twilio is set up)

Twilio needs its own sign-up first: see `docs/phone.md`. Then add the four Twilio secrets and run the cell below.

In [ ]:
#@title PHONE (optional, needs a paid Twilio account): ring one of your numbers
TO = "+234"  #@param {type:"string"}
needed = ["TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN", "TWILIO_FROM_NUMBER", "ALLOWED_NUMBERS"]
if not all(secret(n) for n in needed):
    raise SystemExit("Add these Colab Secrets first: " + ", ".join(needed))
for n in needed:
    set_env(n, secret(n))
!uv run modal secret create nudge-env --from-dotenv .env --force
!uv run modal deploy deploy/api.py

import requests
env = read_env()
r = requests.post(env["API_URL"] + "/phone/calls", headers={"X-API-Key": env["API_KEY"]}, json={"to": TO}, timeout=60)
print(r.status_code, r.json())

## When all of this is green: hand over to GitHub Actions

1. In the `nudge-engine` repo on GitHub: Settings, Secrets and variables, Actions, and add `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `HF_TOKEN` and `MODELS_API_KEY`.
2. Push to `main`. The tests run, then the voice, the brain and the API redeploy by themselves. Nothing runs on a timer.
3. In Vercel, set `ENGINE_URL` (the API URL, saved as `API_URL` in `.env` and printed by SETUP) and `ENGINE_API_KEY` (your `API_KEY`).

The settings secret `nudge-env` stays something you make from this notebook. Run SETUP again whenever you change a key.

To read the API's logs at any time: `!uv run modal app logs nudge-api`.